# 1 · Clean GBIF occurrence data

**Project:** Do species-rich areas in Singapore cluster along green corridors?
Cleaning steps:
1. Load only the columns we need
2. Keep records identified to species level
3. Keep wild observations only (drop zoo animals, fossils, etc.)
4. Remove records with GBIF location problems
5. Keep points inside Singapore with location accuracy ≤ 1 km
6. Remove duplicate records
7. Label the data source (eBird, iNaturalist, other)


In [1]:
import glob, os; os.chdir(r"C:\Users\teohj\OneDrive\Documents\GitHub\sg-wildlife-corridors\notebooks") #glob finds files whose name matches a pattern, os works with folders and files
import pandas as pd

pd.set_option("display.max_columns", 30) #Show XX columns
RAW_DIRS = "../data/raw"
PROCESSED_DIRS = "../data/processed"
os.makedirs(PROCESSED_DIRS, exist_ok = True) #creates the /data/processed folder if missing, dont raise error if folder already exist

csv_files = glob.glob(os.path.join(RAW_DIRS, "*.csv")) #os.path.join finds the raw folder, glob.glob finds the file path (.csv)
RAW_FILE = max(csv_files, key=os.path.getsize) #key=os.path.getsize compares csv files by size
#since the GBIF file is the biggest one, max() finds it automatically without needing to type its long name

#confirm the file found
print("Using:", RAW_FILE, f"({os.path.getsize(RAW_FILE)/ 1e6:.0f} MB)")
#/1e6 converts to megabytes by dividing bytes by 1,000,000
#:.0f rounds to 0 dp

Using: ../data/raw\0008859-261110092616502.csv (1139 MB)


# 2 · Load columns


In [2]:
df = pd.read_csv(RAW_FILE, sep="\t") #GBIF files are separated by tabs sep="\t"
df.info(show_counts=True)

C:\Users\teohj\AppData\Local\Temp\ipykernel_40228\618296739.py:1: DtypeWarning: Columns (4,5,6,7,8,9,10,14,16,34,38,39,40,41,43,45,48) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(RAW_FILE, sep="\t") #GBIF files are separated by tabs sep="\t"


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2002868 entries, 0 to 2002867
Data columns (total 50 columns):
 #   Column                            Non-Null Count    Dtype  
---  ------                            --------------    -----  
 0   gbifID                            2002868 non-null  int64  
 1   datasetKey                        2002868 non-null  object 
 2   occurrenceID                      2002419 non-null  object 
 3   kingdom                           2002868 non-null  object 
 4   phylum                            1995312 non-null  object 
 5   class                             1994095 non-null  object 
 6   order                             1985951 non-null  object 
 7   family                            1993467 non-null  object 
 8   genus                             1991680 non-null  object 
 9   species                           1979689 non-null  object 
 10  infraspecificEpithet              27374 non-null    object 
 11  taxonRank                         200

In [3]:
pd.set_option("display.max_columns", None)
df.describe(include="all")

,gbifID,datasetKey,occurrenceID,kingdom,phylum,class,order,family,genus,species,infraspecificEpithet,taxonRank,scientificName,verbatimScientificName,verbatimScientificNameAuthorship,countryCode,locality,stateProvince,occurrenceStatus,individualCount,publishingOrgKey,decimalLatitude,decimalLongitude,coordinateUncertaintyInMeters,coordinatePrecision,elevation,elevationAccuracy,depth,depthAccuracy,eventDate,day,month,year,taxonKey,speciesKey,basisOfRecord,institutionCode,collectionCode,catalogNumber,recordNumber,identifiedBy,dateIdentified,license,rightsHolder,recordedBy,typeStatus,establishmentMeans,lastInterpreted,mediaType,issue
count,2.002868e+06,2002868,2002419,2002868,1995312,1994095,1985951,1993467,1991680,1979689,27374,2002868,2002868,2002868,6022,2002868,1485063,1977465,2002868,1.386272e+06,2002868,2.002868e+06,2.002868e+06,4.403160e+05,13.00000,524.000000,435.0,167.000000,125.000000,2002868,2.002846e+06,2.002863e+06,2.002868e+06,2002868,1979676,2002868,1973208,1974033,1974643,5536,509729,508984,2002868,536552,2002225,83,0.0,2002868,505177,1987410
unique,NaN,63,2002419,1,14,43,201,1028,3999,6098,365,18,7566,7897,503,1,15142,18,1,NaN,38,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,468277,NaN,NaN,NaN,7566,6096,6,57,37,1974495,5528,8567,440593,3,8033,16536,3,NaN,1827683,65,110
top,NaN,4fa7b334-ce0d-4e88-aaae-2e0c138d049e,45438791435F983FA6D9F9D56230F894.mc.7D823CDA43...,Animalia,Chordata,Aves,Passeriformes,Columbidae,Pycnonotus,Acridotheres javanicus,macromaculatus,SPECIES,"Acridotheres javanicus Cabanis, 1851",Acridotheres javanicus,"(Linnaeus, 1758)",SG,L952084,Singapore,DETECTED,NaN,e2e717bf-551a-4917-bdc9-4fa0f342c530,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2020-12-06,NaN,NaN,NaN,9KHR,9KHR,HUMAN_OBSERVATION,CLO,EBIRD,ENT00000730,GY-SG16-RhoG; ZRC HYM 00000290,big-simonchan,2019-01-01T00:00:00,CC_BY_4_0,big-simonchan,big-simonchan,Paratype,NaN,2026-11-10T13:53:34.209Z,StillImage,COORDINATE_ROUNDED;GEODETIC_DATUM_ASSUMED_WGS8...
freq,NaN,1451881,1,2002868,1686644,1612621,750838,158699,81549,62374,5710,1904922,62374,62373,544,2002868,102572,1459390,2002868,NaN,1451881,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3518,NaN,NaN,NaN,62374,62374,1993222,1451881,1451881,21,3,63199,99,1492397,66638,66638,39,NaN,11,358243,1119418
mean,4.544809e+09,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.812600e+00,NaN,1.355064e+00,1.038212e+02,4.229161e+03,0.00001,74.610687,0.0,5.807784,0.408000,NaN,1.557685e+01,6.626003e+00,2.022065e+03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
std,1.462980e+09,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.629958e+01,NaN,5.457232e-02,7.738986e-02,1.148473e+05,0.00000,22.303254,0.0,5.705345,0.613635,NaN,8.896244e+00,3.622812e+00,2.703779e+00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
min,1.065600e+09,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.000000e+00,NaN,1.000239e+00,1.035333e+02,1.000000e+00,0.00001,0.000000,0.0,0.000000,0.000000,NaN,1.000000e+00,1.000000e+00,2.015000e+03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25%,3.456310e+09,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.000000e+00,NaN,1.314832e+00,1.037718e+02,1.700000e+01,0.00001,58.000000,0.0,0.500000,0.000000,NaN,8.000000e+00,3.000000e+00,2.020000e+03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
50%,4.607436e+09,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.000000e+00,NaN,1.351373e+00,1.038160e+02,2.050000e+02,0.00001,91.000000,0.0,4.000000,0.000000,NaN,1.500000e+01,7.000000e+00,2.022000e+03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
75%,5.606690e+09,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.000000e+00,NaN,1.401239e+00,1.038681e+02,2.980000e+02,0.00001,96.000000,0.0,12.000000,0.500000,NaN,2.300000e+01,1.000000e+01,2.024000e+03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN

In [4]:
df.isnull().mean().sort_values(ascending=False) *100

establishmentMeans                  100.000000
coordinatePrecision                  99.999351
typeStatus                           99.995856
depthAccuracy                        99.993759
depth                                99.991662
elevationAccuracy                    99.978281
elevation                            99.973838
recordNumber                         99.723596
verbatimScientificNameAuthorship     99.699331
infraspecificEpithet                 98.633260
coordinateUncertaintyInMeters        78.015725
mediaType                            74.777319
dateIdentified                       74.587242
identifiedBy                         74.550045
rightsHolder                         73.210816
individualCount                      30.785653
locality                             25.853177
institutionCode                       1.480876
collectionCode                        1.439685
catalogNumber                         1.409229
stateProvince                         1.268331
speciesKey   

In [5]:
df.nunique().sort_values()

establishmentMeans                        0
coordinatePrecision                       1
elevationAccuracy                         1
kingdom                                   1
occurrenceStatus                          1
countryCode                               1
typeStatus                                3
license                                   3
depthAccuracy                             3
basisOfRecord                             6
year                                     12
month                                    12
phylum                                   14
stateProvince                            18
taxonRank                                18
depth                                    19
elevation                                27
day                                      31
collectionCode                           37
publishingOrgKey                         38
class                                    43
institutionCode                          57
datasetKey                      

In [6]:
#df = df.drop(columns = ["establishmentMeans", "coordinatePrecision", "typeStatus", "depthAccuracy",
                       #"depth", "elevationAccuracy", "elevation", "recordNumber", "verbatimScientificNameAuthorship",
                       #"infraspecificEpithet", "coordinateUncertaintyInMeters", "mediaType", "dateIdentified",
                       #"identifiedBy", "rightsHolder", "individualCount", "locality", "kingdom", "occurrenceStatus",
                       #"countryCode"])
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2002868 entries, 0 to 2002867
Data columns (total 50 columns):
 #   Column                            Dtype  
---  ------                            -----  
 0   gbifID                            int64  
 1   datasetKey                        object 
 2   occurrenceID                      object 
 3   kingdom                           object 
 4   phylum                            object 
 5   class                             object 
 6   order                             object 
 7   family                            object 
 8   genus                             object 
 9   species                           object 
 10  infraspecificEpithet              object 
 11  taxonRank                         object 
 12  scientificName                    object 
 13  verbatimScientificName            object 
 14  verbatimScientificNameAuthorship  object 
 15  countryCode                       object 
 16  locality                          ob

In [7]:
df.describe(include = "all")

,gbifID,datasetKey,occurrenceID,kingdom,phylum,class,order,family,genus,species,infraspecificEpithet,taxonRank,scientificName,verbatimScientificName,verbatimScientificNameAuthorship,countryCode,locality,stateProvince,occurrenceStatus,individualCount,publishingOrgKey,decimalLatitude,decimalLongitude,coordinateUncertaintyInMeters,coordinatePrecision,elevation,elevationAccuracy,depth,depthAccuracy,eventDate,day,month,year,taxonKey,speciesKey,basisOfRecord,institutionCode,collectionCode,catalogNumber,recordNumber,identifiedBy,dateIdentified,license,rightsHolder,recordedBy,typeStatus,establishmentMeans,lastInterpreted,mediaType,issue
count,2.002868e+06,2002868,2002419,2002868,1995312,1994095,1985951,1993467,1991680,1979689,27374,2002868,2002868,2002868,6022,2002868,1485063,1977465,2002868,1.386272e+06,2002868,2.002868e+06,2.002868e+06,4.403160e+05,13.00000,524.000000,435.0,167.000000,125.000000,2002868,2.002846e+06,2.002863e+06,2.002868e+06,2002868,1979676,2002868,1973208,1974033,1974643,5536,509729,508984,2002868,536552,2002225,83,0.0,2002868,505177,1987410
unique,NaN,63,2002419,1,14,43,201,1028,3999,6098,365,18,7566,7897,503,1,15142,18,1,NaN,38,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,468277,NaN,NaN,NaN,7566,6096,6,57,37,1974495,5528,8567,440593,3,8033,16536,3,NaN,1827683,65,110
top,NaN,4fa7b334-ce0d-4e88-aaae-2e0c138d049e,45438791435F983FA6D9F9D56230F894.mc.7D823CDA43...,Animalia,Chordata,Aves,Passeriformes,Columbidae,Pycnonotus,Acridotheres javanicus,macromaculatus,SPECIES,"Acridotheres javanicus Cabanis, 1851",Acridotheres javanicus,"(Linnaeus, 1758)",SG,L952084,Singapore,DETECTED,NaN,e2e717bf-551a-4917-bdc9-4fa0f342c530,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2020-12-06,NaN,NaN,NaN,9KHR,9KHR,HUMAN_OBSERVATION,CLO,EBIRD,ENT00000730,GY-SG16-RhoG; ZRC HYM 00000290,big-simonchan,2019-01-01T00:00:00,CC_BY_4_0,big-simonchan,big-simonchan,Paratype,NaN,2026-11-10T13:53:34.209Z,StillImage,COORDINATE_ROUNDED;GEODETIC_DATUM_ASSUMED_WGS8...
freq,NaN,1451881,1,2002868,1686644,1612621,750838,158699,81549,62374,5710,1904922,62374,62373,544,2002868,102572,1459390,2002868,NaN,1451881,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3518,NaN,NaN,NaN,62374,62374,1993222,1451881,1451881,21,3,63199,99,1492397,66638,66638,39,NaN,11,358243,1119418
mean,4.544809e+09,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.812600e+00,NaN,1.355064e+00,1.038212e+02,4.229161e+03,0.00001,74.610687,0.0,5.807784,0.408000,NaN,1.557685e+01,6.626003e+00,2.022065e+03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
std,1.462980e+09,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.629958e+01,NaN,5.457232e-02,7.738986e-02,1.148473e+05,0.00000,22.303254,0.0,5.705345,0.613635,NaN,8.896244e+00,3.622812e+00,2.703779e+00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
min,1.065600e+09,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.000000e+00,NaN,1.000239e+00,1.035333e+02,1.000000e+00,0.00001,0.000000,0.0,0.000000,0.000000,NaN,1.000000e+00,1.000000e+00,2.015000e+03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25%,3.456310e+09,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.000000e+00,NaN,1.314832e+00,1.037718e+02,1.700000e+01,0.00001,58.000000,0.0,0.500000,0.000000,NaN,8.000000e+00,3.000000e+00,2.020000e+03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
50%,4.607436e+09,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.000000e+00,NaN,1.351373e+00,1.038160e+02,2.050000e+02,0.00001,91.000000,0.0,4.000000,0.000000,NaN,1.500000e+01,7.000000e+00,2.022000e+03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
75%,5.606690e+09,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.000000e+00,NaN,1.401239e+00,1.038681e+02,2.980000e+02,0.00001,96.000000,0.0,12.000000,0.500000,NaN,2.300000e+01,1.000000e+01,2.024000e+03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN

In [8]:
df = df.drop(columns=["occurrenceID"]) #drop occurenceID because is has same meaning as  gbifID

In [9]:
df = df.drop(columns= ["phylum", "genus", "taxonRank"])

In [10]:
df = df.drop(columns=["scientificName", "verbatimScientificName"])

In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2002868 entries, 0 to 2002867
Data columns (total 44 columns):
 #   Column                            Dtype  
---  ------                            -----  
 0   gbifID                            int64  
 1   datasetKey                        object 
 2   kingdom                           object 
 3   class                             object 
 4   order                             object 
 5   family                            object 
 6   species                           object 
 7   infraspecificEpithet              object 
 8   verbatimScientificNameAuthorship  object 
 9   countryCode                       object 
 10  locality                          object 
 11  stateProvince                     object 
 12  occurrenceStatus                  object 
 13  individualCount                   float64
 14  publishingOrgKey                  object 
 15  decimalLatitude                   float64
 16  decimalLongitude                  fl

In [12]:
df = df.drop(columns=["day"])

In [13]:
df = df.drop(columns=["collectionCode", "catalogNumber", "gbifID"])

In [14]:
df = df.drop(columns=["publishingOrgKey", "taxonKey", "speciesKey", "license", "lastInterpreted"])

In [15]:
df = df.drop(columns=["lastInterpreted"])

KeyError: "['lastInterpreted'] not found in axis"

In [ ]:
df = df.drop(columns=["datasetKey"])

In [ ]:
df.value_counts("stateProvince")

In [ ]:
df.value_counts("taxonRank")

In [ ]:
df.info()

In [ ]:
df = df.drop(columns=["stateProvince"])

In [ ]:
log = [("Raw records", len(df))]
print(log)

In [ ]:
df = df.dropna(subset=["species"]) #removes rows if null species
log.append(("No. species records", len(df))) 
log = log[:2]
print(log)

In [ ]:
df.value_counts("basisOfRecord")

In [ ]:
#since we want to find out species biodiveristy in green spaces we only want to use wild sightings
#hence we only use human and machine observations filtering out samples and specimens
keep = ["HUMAN_OBSERVATION", "MACHINE_OBSERVATION", "OBSERVATION"]
df = df[df["basisOfRecord"].isin(keep)]
log.append(("Wild observation records", len(df)))
log = log[:3]
print(log)

In [ ]:
df.value_counts("issue").head(7)

In [ ]:
df["issue"].str.split(";").explode().unique() #.explode() give each item its own row

In [ ]:
df[["decimalLatitude", "decimalLongitude"]].describe()

In [ ]:
valid_lat = df["decimalLatitude"].between(1.15, 1.483)
valid_long = df["decimalLongitude"].between(103.6, 104.417)
df = df[valid_lat & valid_long]
log.append(("Within SG's coordinates:", len(df)))
log = log[:4]
print(log)

In [ ]:
df.info()

In [ ]:
#Removing duplicate entries
same_record = ["decimalLatitude", "decimalLongitude", "eventDate", "recordedBy", "species"]
df = df.drop_duplicates(subset=same_record)
log.append(("Duplicates removed: ", len(df)))
log = log[:5]
print(log)

In [ ]:
df["observerID"] = df["recordedBy"].factorize()[0]
df = df.drop(columns="recordedBy")

In [ ]:
df["observerID"].describe()

In [ ]:
summary = pd.DataFrame(log, columns=["Step", "Records left"])
summary["Removed"] = summary["Records left"].shift() - summary["Records left"]
summary

In [ ]:
!pip install pyarrow

In [ ]:
df.to_parquet(os.path.join(PROCESSED_DIRS, "gbif_clean.parquet"), index = False)


In [16]:
df.to_csv(os.path.join(PROCESSED_DIRS, "gbif_clean_for_qgis.csv"), index=False)